# Compare policies

*Author: Mohammad Al-Jarrah*

Reads the evaluation files of `results/<task>/` (written by `scripts/evaluate.py` or `scripts/reproduce_paper.sh`) and plots the mean score against the time per episode, as in Figure 1 of the paper. Policies without results are skipped.

In [ ]:
import os, sys, json
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')            # headless rendering of Push-T
ROOT = os.path.abspath('..')                                  # the repository root
sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as ShowImage, display

import experiment as X
import visualization as V
from policies import NAMES
fontsize = 11
labeling = True                                               # set False to hide axis labels

In [ ]:
TASK = 'pusht'      # pusht | square
ENTRIES = [('OTP', 'otp', ''), ('FMP', 'fmp', ''), ('SIP', 'sip', ''), ('KRP', 'krp', ''),
           ('DP', 'dp', ''), ('DPR', 'dpr', ''), ('CP 1-step', 'cp', ''), ('CP 3-step', 'cp', '_3step')]
SUFFIX = ''         # Square: '' (plain), '_mean8', '_noise' or '_noise_mean8'
PROPOSED = {'OTP', 'FMP', 'SIP'}

rows = []
for label, policy, steps in ENTRIES:
    files = sorted(__import__('glob').glob(os.path.join(ROOT, 'results', TASK, '%s%s%s_seed*.json' % (policy, SUFFIX, steps))))
    if not files:
        continue
    runs = [json.load(open(f)) for f in files]
    s = X.summarize(np.concatenate([r['score'] for r in runs]))
    rows.append((label, s['mean'], s['se'], np.mean([r['seconds_per_episode'] for r in runs]), len(runs)))
    print('%-10s score %.3f +- %.3f  %.2f s/episode  (%d seeds)' % rows[-1])

In [ ]:
plt.figure(figsize=(5, 3.5))
for label, mean, se, sec, _ in rows:
    color = 'C3' if label in PROPOSED else 'C0'
    plt.errorbar(sec, mean, yerr=se, fmt='o', color=color, ms=6, capsize=2)
    plt.annotate(label, (sec, mean), textcoords='offset points', xytext=(5, 4), fontsize=fontsize - 2)
plt.xscale('log')
if labeling: plt.xlabel('time per episode [s]', fontsize=fontsize)
if labeling: plt.ylabel('score' if TASK == 'pusht' else 'success rate', fontsize=fontsize)
plt.title('%s: proposed (red) and Gaussian-reference (blue) policies' % TASK, fontsize=fontsize)
plt.tight_layout(); plt.show()